# RawTreeWriter_sm3: 同期した TPC の生データの TTree

`ReadData` がボード間でそろえたイベントを、**生データのまま 1 イベント 1 行**の TTree `RawTreeWriter_sm3/raw` に書きます。
一度書いておけば、ゲインマップ・エネルギーの較正・ドリフト速度などを、生データを読み直さずに作り直せます(EricaNext_try の `4_MakeCalibration` と同じ使い方)。
`ReadData` の直後につなぎます(設定ファイル `smile3_raw`)。シンチレータ(PSA)のデータは書きません(`PSA_TreeWriter` を使う)。

| 列 | 型 | 中身 |
|---|---|---|
| `event_id` | uint | イベント ID(トリガーカウンタ `N_trg`) |
| `sevt_id` | uint | `ReadData` のイベントの番号(0 から) |
| `loopback` | int | ボードの読み進めが起きたイベント(`IDcheck_Loopback`)なら 1 |
| `anode_time`, `cathode_time` | vector&lt;uint&gt; | ボードごとの `N_time`(`run_info` の `anode_ids` の順) |
| `hit_a_strip`, `hit_a_clock`, `hit_c_strip`, `hit_c_clock` | vector&lt;short&gt; | ストリップ(0〜1535)× クロックのヒット(`save_hits`) |
| `rec_a_board`, `rec_a_clock`, `rec_c_board`, `rec_c_clock` | vector&lt;short&gt; | エンコーダーの記録(ボード、クロック)。ストリップのビットが全部 0 の記録も含む |
| `fadc_ch`, `fadc_wave` | vector&lt;short&gt; | データのあるボードの FADC: ch(0〜47 アノード 4 × ボード + ch、48〜95 カソード)と、その順に 512 サンプルずつ。カソードは反転済み(1023 − ADC)。`save_threshold` > 0 なら、ベースラインからの最大の高さがそれを超える ch だけ(`save_fadc`) |

`RawTreeWriter_sm3/run_info` には、ボードの番号(`anode_ids`, `cathode_ids`)、FADC のサンプル数、書いたときの設定が 1 行入っています。

In [ ]:
# ===== 入力と出力(絶対パス。データ量が大きいので、リポジトリの外の NAS などに置く)=====
from pathlib import Path
# 入力: TPC の生データのランのディレクトリ
RUN_DIR = Path("/mnt/nas_eureka/onoda/<書き換え>/run_20260623T072649Z")
# 出力: チェーンの出力の置き場(チュートリアル間で共有)
CACHE = Path("/mnt/nas_eureka/onoda/<書き換え>/EricaNext_tutorial")

for p in (RUN_DIR, CACHE):
    if "<" in str(p):
        raise ValueError(f"パスを書き換えてください: {p}")
CACHE.mkdir(parents=True, exist_ok=True)
print("入力:", RUN_DIR)
print("出力:", CACHE)


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    dt = np.float32 if h.InheritsFrom("TArrayF") else np.float64   # TH2F / TH2D
    a = np.frombuffer(h.GetArray(), dtype=dt, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2).astype(float)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def th1(h):
    """ROOT の TH1 -> (中身, ビンの中心)"""
    n = h.GetNbinsX()
    return np.array([h.GetBinContent(i) for i in range(1, n + 1)]), np.array([h.GetBinCenter(i) for i in range(1, n + 1)])

def vec(column):
    """RDataFrame.AsNumpy() の配列型の列 -> numpy の配列のリスト"""
    return [np.asarray(v) for v in column]

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-06-23 の TPC の較正データ **`run_20260623T072649Z`**(TPC_cal、アノード 12 枚・カソード 11 枚、約 4 GB、約 228 秒、約 10 万イベント)を使います。
ランのディレクトリ(入力)は冒頭の `RUN_DIR` に絶対パスで書きます。
1 つのディレクトリにアノードとカソードの全ボードが入っているので、`ReadData` がそのまま読めます
(FM のランのように 2 つの storage に分かれているときは、片方を `--data-dir`、もう片方を `--extra-data-dir` で渡す)。

このランは uPIC0〜2 のレートが安定していて、uPIC3 は電荷がほとんど出ていません。最後の約 10 秒は全 uPIC のレートが跳ね上がる(原因は不明。Notion の「TPC 確認図と QL」)ので、
ここでは先頭の 50000 イベント(約 110 秒)を使います。

In [ ]:
print("OK     " if RUN_DIR.is_dir() else "MISSING", RUN_DIR)
meta = json.load(open(RUN_DIR / "run_meta.json"))
print("mode:", meta["mode"], " boards:", len(meta["active_detectors"]))

## 2. チェーンを流す

設定ファイル `smile3_tpc_only`(`ReadData → CalcFADC → TPC_Calibration → TOT_Skewness`)に、図を作るパラメータを足して先頭 50000 イベントを流します(1 分ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_tpc_only -o tpc_20260623.root -n 50000 --data-dir <RUN_DIR> \
    --set TPC_Calibration.Image_XY_TH=true --set TPC_Calibration.Spectrum_TH=true
```

生の波形とヒットは、`smile3_raw`(`ReadData → RawTreeWriter_sm3`)で先頭 5000 イベントをツリーに書いたものから読みます。
出力は冒頭の `CACHE` に置き、既にあれば流し直しません(TPC のチュートリアルで共有)。

In [ ]:
def run_chain(config, out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)。ログを返す"""
    cmd = [sys.executable, str(RUN_CHAIN), config, "-o", str(out), "--data-dir", str(RUN_DIR), *args]
    print(" ".join(cmd))
    if out.exists():
        return ""
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
    (out.parent / (out.name + ".log")).write_text(r.stdout)
    return r.stdout

OUT = CACHE / "tpc_20260623.root"
run_chain("smile3_tpc_only", OUT, "-n", "50000",
          "--set", "TPC_Calibration.Image_XY_TH=true", "--set", "TPC_Calibration.Spectrum_TH=true")
OUT_RAW = CACHE / "tpc_20260623_raw.root"
run_chain("smile3_raw", OUT_RAW, "-n", "5000")
F = ROOT.TFile(str(OUT))

def evs_table(log_path):
    """チェーンのログから、イベント選択(evs)の数の表を取り出す"""
    lines = open(log_path).read().split("Result of Event Selections")[1].splitlines()
    rows = [l.split() for l in lines if len(l.split()) == 3 and l.split()[1].isdigit()]
    return {r[0]: int(r[1]) for r in rows}

## 3. 中身

列と型、`run_info` を表示します。

In [ ]:
f = ROOT.TFile(str(OUT_RAW))
tree = f.Get("RawTreeWriter_sm3/raw")
print("entries:", tree.GetEntries(), f" file {OUT_RAW.stat().st_size / 1e6:.0f} MB")
for br in tree.GetListOfBranches():
    print(f"  {br.GetName():14s} {br.GetClassName() or br.GetTitle()}")
info = ROOT.RDataFrame("RawTreeWriter_sm3/run_info", str(OUT_RAW)).AsNumpy(["anode_ids", "cathode_ids", "n_fadc_clk", "save_threshold"])
print("anode_ids:", list(info["anode_ids"][0]), " cathode_ids:", list(info["cathode_ids"][0]), " n_fadc_clk:", info["n_fadc_clk"][0])

## 4. 読み方

`RDataFrame` の `AsNumpy` で読み、配列の列は `np.asarray` で 1 イベントずつ numpy にします。
`fadc_wave` は `reshape(-1, 512)` で ch ごとの波形になります。下は、1 イベントのアノードのヒットと、その FADC の 4 ch を並べたものです
(ch k はストリップ 32k〜32k+31 の和。FADC の 1 サンプルはクロック 2 つと考えられるが、仕様書では未確認)。

In [ ]:
r = ROOT.RDataFrame("RawTreeWriter_sm3/raw", str(OUT_RAW)).AsNumpy(["event_id", "hit_a_strip", "hit_a_clock", "fadc_ch", "fadc_wave"])
for e in range(len(r["event_id"])):
    s = np.asarray(r["hit_a_strip"][e])
    if len(s) > 100 and len(np.unique(s // 128)) == 1:
        break
board = int(s[0] // 128)
chs = np.asarray(r["fadc_ch"][e]); w = np.asarray(r["fadc_wave"][e]).reshape(-1, 512)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(s, np.asarray(r["hit_a_clock"][e]), "r.", ms=2)
for k in range(5):
    ax[0].axvline(128 * board + 32 * k, color="gray", ls=":")
ax[0].set_xlim(128 * board - 4, 128 * (board + 1) + 4); ax[0].set_xlabel("anode strip"); ax[0].set_ylabel("clock")
ax[0].set_title(f"event_id {r['event_id'][e]}: anode board A{board}")
for k in range(4):
    i = np.where(chs == 4 * board + k)[0]
    if len(i):
        ax[1].plot(w[i[0]], lw=0.8, label=f"ch {k} (strips {128 * board + 32 * k}-{128 * board + 32 * k + 31})")
ax[1].set_xlabel("sample"); ax[1].set_ylabel("ADC"); ax[1].legend(fontsize=8); ax[1].set_title("FADC of the board")
fig.tight_layout(); plt.show()

## 5. 大きさ

FADC の波形を全部書くと大きくなります(5000 イベントで上の大きさ)。`save_threshold`(例 20 ADU)を付けると、信号の無い ch を書きません。
ヒットだけでよいときは `save_fadc: false` にします。

## 6. パラメータ

In [ ]:
show_params("smile3_raw", "RawTreeWriter_sm3")